In [1]:
import numpy as np
import pandas as pd

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Layer
from tensorflow.keras.layers import TextVectorization

2026-05-16 09:20:57.054325: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778923257.374551      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778923257.450758      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778923258.053893      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778923258.053942      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778923258.053945      57 computation_placer.cc:177] computation placer alr

In [2]:
# belum pakai dataset asli, masih memakai dataset buatan pribadi
df_inventaris = pd.read_csv('/kaggle/input/datasets/miguelvalentino/test-inventaris/Data_testing.csv')

df_inventaris['fitur_teks_gabungan'] = (
    df_inventaris['nama_barang'] + " " + 
    df_inventaris['deskripsi'] + " " + 
    df_inventaris['kategori']
)

print(df_inventaris[['item_id', 'fitur_teks_gabungan']].head())

   item_id                                fitur_teks_gabungan
0  INV-001  Tenda Camping Besar BlackDog Peralatan Blackdo...
1  INV-002  Tenda Camping Besar NatureHike Tenda camping m...
2  INV-003  BlackDog Auto tent Tenda camping otomatis Blac...
3  INV-004  NatureHike Village 13 Tenda glamping ukuran be...
4  INV-005  Eiger Shira 1P Tenda solo camping ringan dari ...


In [3]:
import random

teks_barang_A = []
teks_barang_B = []
labels = []

kategori_list = df_inventaris['kategori'].unique()
df_per_kategori = {kat: df_inventaris[df_inventaris['kategori'] == kat] for kat in kategori_list}

for _ in range(500):
    kat_valid = [k for k, v in df_per_kategori.items() if len(v) > 1]
    kat = random.choice(kat_valid)
    
    sampel = df_per_kategori[kat].sample(2) 
    teks_barang_A.append(sampel.iloc[0]['fitur_teks_gabungan'])
    teks_barang_B.append(sampel.iloc[1]['fitur_teks_gabungan'])
    labels.append(1.0)

for _ in range(500):
    kat_1, kat_2 = random.sample(list(kategori_list), 2)
    
    item_1 = df_per_kategori[kat_1].sample(1).iloc[0]['fitur_teks_gabungan']
    item_2 = df_per_kategori[kat_2].sample(1).iloc[0]['fitur_teks_gabungan']
    
    teks_barang_A.append(item_1)
    teks_barang_B.append(item_2)
    labels.append(0.0)

In [4]:
teks_barang_A = np.array(teks_barang_A)
teks_barang_B = np.array(teks_barang_B)
labels = np.array(labels, dtype="float32")

indices = np.arange(len(labels))
np.random.shuffle(indices)

teks_barang_A = teks_barang_A[indices]
teks_barang_B = teks_barang_B[indices]
labels = labels[indices]

In [5]:
print(f"Total pasangan data latih: {len(labels)}")
print("\nContoh Pasangan Data Latih (A, B, Label):")
for i in range(3):
    print(f"A    : {teks_barang_A[i]}")
    print(f"B    : {teks_barang_B[i]}")
    print(f"Label: {labels[i]}\n")

Total pasangan data latih: 1000

Contoh Pasangan Data Latih (A, B, Label):
A    : Set Piring NatureHike Piring dan mangkuk plastik food grade untuk camping Alat Masak Outdoor
B    : Teko Camping Consina 1L Ketel air untuk menyeduh kopi atau teh di gunung Alat Masak Outdoor
Label: 1.0

A    : Consina Tarebbi 60L Carrier favorit tangguh untuk pendakian jarak jauh Tas Ransel
B    : Mobi Garden Era 260 Tenda glamping premium bentuk kabin Tenda Outdoor
Label: 0.0

A    : Pasak Tenda NatureHike Alloy Pasak tenda bahan aluminium alloy kuat dan ringan isi 10 Aksesoris Tenda
B    : Tiang Flysheet Arei Tiang penyangga flysheet bisa diatur tingginya Aksesoris Tenda
Label: 1.0



In [6]:
semua_teks = np.concatenate((teks_barang_A, teks_barang_B))

max_vocab_size = 500
max_sequence_len = 20 

In [7]:
vectorize_layer = TextVectorization(
    max_tokens=max_vocab_size,
    output_mode='int',
    output_sequence_length=max_sequence_len
)

vectorize_layer.adapt(semua_teks)

print("Contoh teks asli  :", teks_barang_A[0])
print("Hasil vektorisasi :", vectorize_layer(teks_barang_A[0]).numpy())
print("-" * 50)

Contoh teks asli  : Set Piring NatureHike Piring dan mangkuk plastik food grade untuk camping Alat Masak Outdoor
Hasil vektorisasi : [142 148   7 148  15 237 235 241 238   5   9  24  16   3   0   0   0   0
   0   0]
--------------------------------------------------


2026-05-16 09:21:28.266792: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [8]:
class CosineSimilarityLayer(Layer):
    def __init__(self, **kwargs):
        super(CosineSimilarityLayer, self).__init__(**kwargs)

    def call(self, inputs):
        vec_a, vec_b = inputs
        vec_a = tf.math.l2_normalize(vec_a, axis=1)
        vec_b = tf.math.l2_normalize(vec_b, axis=1)
        return tf.reduce_sum(vec_a * vec_b, axis=1, keepdims=True)

def build_siamese_model():
    input_a = Input(shape=(1,), dtype=tf.string, name="input_barang_target")
    input_b = Input(shape=(1,), dtype=tf.string, name="input_barang_kandidat")

    embedding_layer = Embedding(input_dim=max_vocab_size, output_dim=16)
    pooling_layer = GlobalAveragePooling1D()
    dense_layer = Dense(16, activation='relu')

    def process_tower(text_input):
        x = vectorize_layer(text_input)
        x = embedding_layer(x)
        x = pooling_layer(x)
        x = dense_layer(x)
        return x

    tower_a = process_tower(input_a)
    tower_b = process_tower(input_b)

    similarity_score = CosineSimilarityLayer()([tower_a, tower_b])

    model = Model(inputs=[input_a, input_b], outputs=similarity_score, name="Siamese_Inventory_Model")
    
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    
    return model

In [9]:
model = build_siamese_model()

model.summary()

Model: "Siamese_Inventory_Model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_barang_target │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_barang_kandi… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, 20)        │          0 │ input_barang_tar… │
│ (TextVectorization) │                   │            │ input_barang_kan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 16)    │      8,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │ text_vectorizati… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ embedding[0][0],  │
│ (GlobalAveragePool… │                   │            │ embedding[1][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        272 │ global_average_p… │
│                     │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cosine_similarity_… │ (None, 1)         │          0 │ dense[0][0],      │
│ (CosineSimilarityL… │                   │            │ dense[1][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,272 (32.31 KB)

 Trainable params: 8,272 (32.31 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
tensor_A = tf.constant(teks_barang_A.reshape(-1, 1), dtype=tf.string)
tensor_B = tf.constant(teks_barang_B.reshape(-1, 1), dtype=tf.string)

history = model.fit(
    x=[tensor_A, tensor_B],
    y=labels,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - loss: 0.1499 - mae: 0.2888 - val_loss: 0.0435 - val_mae: 0.1172
Epoch 2/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0291 - mae: 0.0773 - val_loss: 0.0215 - val_mae: 0.0511
Epoch 3/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0150 - mae: 0.0367 - val_loss: 0.0177 - val_mae: 0.0305
Epoch 4/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0215 - mae: 0.0326 - val_loss: 0.0146 - val_mae: 0.0343
Epoch 5/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0157 - mae: 0.0339 - val_loss: 0.0175 - val_mae: 0.0309
Epoch 6/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0122 - mae: 0.0213 - val_loss: 0.0131 - val_mae: 0.0247
Epoch 7/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0131 - mae: 0.0240 - val_loss: 0.0146 - val_mae: 0.0294
Epoch 8/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0047 - mae: 0.0177 - val_loss: 0.0062 - val_mae: 0.0132
Epoch 9/10
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0090 - mae:

In [12]:
loaded_model = tf.keras.models.load_model(
    nama_file_model,
    custom_objects={'CosineSimilarityLayer': CosineSimilarityLayer}
)

In [13]:
target_item_teks = df_inventaris.iloc[20]['fitur_teks_gabungan'] #ganti angka buat tes barang lainnya
target_item_nama = df_inventaris.iloc[20]['nama_barang']

In [14]:
kandidat_teks = df_inventaris['fitur_teks_gabungan'].values
kandidat_nama = df_inventaris['nama_barang'].values

In [15]:
target_array = np.array([target_item_teks] * len(kandidat_teks))
tensor_target = tf.constant(target_array.reshape(-1, 1), dtype=tf.string)
tensor_kandidat = tf.constant(kandidat_teks.reshape(-1, 1), dtype=tf.string)

In [16]:
skor_prediksi = loaded_model.predict([tensor_target, tensor_kandidat], verbose=0)


In [17]:
hasil_rekomendasi = []
for i in range(len(kandidat_nama)):
    hasil_rekomendasi.append((kandidat_nama[i], skor_prediksi[i][0]))

hasil_rekomendasi.sort(key=lambda x: x[1], reverse=True)

print(f"Barang yang baru diupdate: **{target_item_nama}**")
print("Rekomendasi barang terkait untuk dicek/diupdate:\n")

for nama, skor in hasil_rekomendasi[1:4]:
    print(f"- {nama} (Skor Kemiripan: {skor:.4f})")

Barang yang baru diupdate: **Bantal Tiup BlackDog**
Rekomendasi barang terkait untuk dicek/diupdate:

- Dhaulagiri Inflatable Mat (Skor Kemiripan: 1.0000)
- Bantal Tiup BlackDog (Skor Kemiripan: 1.0000)
- Matras Spons Eiger (Skor Kemiripan: 1.0000)


In [18]:
model.save('sistemrekomendasi2_moneytor.keras')